# Super Mario Bros Gameplay Training with ConvNeXt-RWKV7 Gamepad

Train the **ConvNeXt-RWKV7 Gamepad** model on the **Super Mario Bros World Model** dataset (`DylanRiden/smb-worldmodel-data`) using **PyTorch Lightning** on Kaggle **2x T4 GPUs** (DDP with mixed precision).

### Pipeline Architecture
1. **Input Normalization:** `_InputNormalize` (DINOv3 ImageNet mean/std)
2. **Spatial Pooling:** `AdaptiveLearnedPool2d` (downsamples inputs to 224x224)
3. **Vision Backbone:** `ConvNeXt-Tiny` with pre-trained **DINOv3** weights (`facebook/dinov3-convnext-tiny-pretrain-lvd1689m`)
4. **Spatial Aggregation:** `LearnedWeightedGAP` (spatial attention + global average pooling)
5. **Temporal Convolution:** `CausalConv1d` with residual shortcut
6. **Recurrent Reasoning:** 4x `RWKV-7` (Goose) linear attention blocks
7. **Gamepad Head:** 21-D output (17 boolean buttons via BCE loss + 4 joystick axes in [-1.0, 1.0] via MSE loss)


In [ ]:
# Install the ConvNeXt Platform package directly from GitHub
%pip install -q "git+https://github.com/Gabz4200/ConvNeXt_Platform.git"


In [ ]:
import os
from functools import partial
import torch
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping, RichProgressBar
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

# Allow safe unpickling for PyTorch 2.6+
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")

# Set global seed for reproducibility
L.seed_everything(42, workers=True)

# Check hardware accelerators (2x T4 on Kaggle)
num_gpus = torch.cuda.device_count()
print(f"PyTorch: {torch.__version__}")
print(f"Lightning: {L.__version__}")
print(f"Available GPUs: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


In [ ]:
from src.data.smb_datamodule import SMBDataModule

# Instantiate Super Mario Bros DataModule
# Downloads smb_frames.zip automatically from DylanRiden/smb-worldmodel-data on Hugging Face Hub
datamodule = SMBDataModule(
    data_dir="data/smb",
    repo_id="DylanRiden/smb-worldmodel-data",
    batch_size=64,             # Per-GPU batch size (effective batch size = 128 on 2x T4)
    val_ratio=0.1,
    test_ratio=0.1,
    image_size=(224, 224),
    target_mode="gamepad_21",  # Maps 8 NES buttons into standard 21-D gamepad targets
    num_workers=2,             # 2 DataLoader workers per GPU (optimal for Kaggle 4-vCPU)
    pin_memory=torch.cuda.is_available(),
    download=True,
    seed=42,
)

# Download and extract dataset before multi-GPU DDP spawning
datamodule.prepare_data()
datamodule.setup("fit")
print(f"Training samples:   {len(datamodule.data_train)}")
print(f"Validation samples: {len(datamodule.data_val)}")


In [ ]:
from src.models.components.convnext_rwkv7 import ConvNeXtRWKV7Gamepad
from src.models.convnext_rwkv7_module import ConvNeXtRWKV7GamepadLitModule

# 1. Neural network backbone combining DINOv3 ConvNeXt and RWKV-7
net = ConvNeXtRWKV7Gamepad(
    in_chans=3,
    convnext_size="tiny",
    pretrained_dinov3=True,     # Loads facebook/dinov3-convnext-tiny-pretrain-lvd1689m
    freeze_convnext=True,       # Freezes ConvNeXt while allowing backprop to AdaptiveLearnedPool2d
    rwkv_dim=256,
    rwkv_head_size=64,
    rwkv_layers=4,
    head_hidden_dim=256,
    num_buttons=17,
    num_joysticks=2,
)

# 2. Optimizer & Cosine Annealing Learning Rate Scheduler
max_epochs = 15
optimizer_factory = partial(torch.optim.AdamW, lr=1e-3, weight_decay=0.01)
scheduler_factory = partial(torch.optim.lr_scheduler.CosineAnnealingLR, T_max=max_epochs, eta_min=1e-6)

# 3. LightningModule wrapper with combined BCE + MSE Gamepad loss and metrics
model = ConvNeXtRWKV7GamepadLitModule(
    net=net,
    optimizer=optimizer_factory,
    scheduler=scheduler_factory,
    joystick_loss_weight=1.0,
)


In [ ]:
# Setup multi-GPU DDP Trainer for Kaggle 2x T4
# Ref: https://lightning.ai/docs/pytorch/stable/reference/common/notebooks
strategy = "ddp_notebook" if num_gpus > 1 else "auto"
devices = num_gpus if num_gpus > 0 else "auto"
accelerator = "gpu" if num_gpus > 0 else "cpu"
precision = "16-mixed" if num_gpus > 0 else "32-true"

logger = CSVLogger(save_dir="logs", name="smb_gamepad")
callbacks = [
    ModelCheckpoint(
        dirpath="checkpoints/smb_gamepad",
        filename="smb-{epoch:02d}-{val/loss:.4f}",
        monitor="val/loss",
        mode="min",
        save_top_k=2,
        save_last=True,
    ),
    EarlyStopping(
        monitor="val/loss",
        patience=4,
        mode="min",
    ),
    RichProgressBar(),
]

trainer = L.Trainer(
    accelerator=accelerator,
    devices=devices,
    strategy=strategy,
    precision=precision,
    max_epochs=max_epochs,
    callbacks=callbacks,
    logger=logger,
    log_every_n_steps=50,
    gradient_clip_val=1.0,
)


In [ ]:
# Fit the model on the Super Mario Bros dataset
trainer.fit(model=model, datamodule=datamodule)

# Test evaluation using best checkpoint
trainer.test(model=model, datamodule=datamodule, ckpt_path="best")


In [ ]:
# Real-Time Online Recurrent Streaming Demo (Frame-by-Frame Inference)
model.eval()
device = next(model.parameters()).device

# Initialize persistent streaming state
streaming_state = model.net.init_streaming_state(batch_size=1, device=device)

# Load test frame from datamodule
datamodule.setup("test")
test_frame, target_gamepad = datamodule.data_test[0]
input_tensor = test_frame.unsqueeze(0).to(device)  # Shape: (1, 3, 224, 224)

# Execute O(1) recurrent step update
with torch.no_grad():
    (full_gamepad, buttons_logits, joysticks), streaming_state = model.net.step(input_tensor, streaming_state)

btn_probs = buttons_logits.sigmoid().squeeze(0).cpu().tolist()
joy_axes = joysticks.squeeze(0).cpu().tolist()

print(f"Predicted 21-D Gamepad Vector Shape: {full_gamepad.shape}")
print("Button Probabilities (first 8 mapped):", [round(p, 3) for p in btn_probs[:8]])
print(f"Predicted Left Stick (X, Y): ({joy_axes[0]:.3f}, {joy_axes[1]:.3f})")
print(f"Target Action Vector (21-D): {target_gamepad.shape}")
